# AIMO3 Dataset Curation & Quality Metrics

Analyze math problems for fine-tuning suitability based on:
1. **Difficulty** - Pass rate, solution complexity
2. **Diversity** - Topic coverage, problem types
3. **Quality** - Solution patterns, tool usage
4. **Novelty** - Uniqueness, deduplication

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from collections import Counter
from pathlib import Path
import hashlib

## 1. Load Dataset

Load the AIMO3 TIR dataset (141k samples)

In [ ]:
# Load the TIR dataset
tir_path = '/kaggle/input/aimo3-tool-integrated-reasoning/Data-Ready/data.csv'
df = pd.read_csv(tir_path)
print(f"Loaded {len(df)} samples")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Quality Metrics

### 2.1 Difficulty Metrics

In [ ]:
def compute_difficulty_metrics(row):
    """Compute difficulty-related metrics for a problem."""
    prompt = str(row.get('prompt', ''))
    completion = str(row.get('completion', ''))
    
    metrics = {}
    
    # 1. Solution length (proxy for difficulty)
    metrics['completion_length'] = len(completion)
    metrics['completion_tokens_approx'] = len(completion.split())
    
    # 2. Number of tool calls (more = harder)
    tool_calls = len(re.findall(r'to=python', completion, re.IGNORECASE))
    metrics['n_tool_calls'] = tool_calls
    
    # 3. Problem text length
    metrics['prompt_length'] = len(prompt)
    
    # 4. Mathematical complexity indicators
    math_keywords = ['integral', 'derivative', 'limit', 'sum', 'product', 'series',
                     'matrix', 'determinant', 'eigenvalue', 'polynomial', 'modular',
                     'congruent', 'prime', 'factorial', 'binomial', 'probability']
    metrics['math_complexity'] = sum(1 for kw in math_keywords if kw.lower() in prompt.lower())
    
    # 5. Numerical answer extraction (if boxed)
    boxed_match = re.search(r'\\boxed\{([^}]+)\}', completion)
    if boxed_match:
        try:
            answer = int(boxed_match.group(1).replace(',', '').strip())
            metrics['answer'] = answer
            metrics['answer_magnitude'] = len(str(abs(answer)))
        except:
            metrics['answer'] = None
            metrics['answer_magnitude'] = 0
    else:
        metrics['answer'] = None
        metrics['answer_magnitude'] = 0
    
    return metrics

# Compute for all rows
print("Computing difficulty metrics...")
difficulty_df = df.apply(compute_difficulty_metrics, axis=1, result_type='expand')
df = pd.concat([df, difficulty_df], axis=1)
print("Done!")
df[['completion_length', 'n_tool_calls', 'math_complexity', 'answer_magnitude']].describe()

### 2.2 Topic Diversity Metrics

In [ ]:
def classify_topic(prompt):
    """Classify problem topic based on keywords."""
    prompt_lower = prompt.lower()
    
    topics = {
        'number_theory': ['prime', 'divisor', 'modulo', 'congruent', 'gcd', 'lcm', 'factor',
                          'digit', 'remainder', 'coprime', 'euler', 'fermat'],
        'algebra': ['equation', 'polynomial', 'root', 'solve', 'variable', 'coefficient',
                    'quadratic', 'cubic', 'linear', 'function', 'expression'],
        'geometry': ['triangle', 'circle', 'angle', 'area', 'perimeter', 'point', 'line',
                     'segment', 'polygon', 'radius', 'diameter', 'tangent', 'chord'],
        'combinatorics': ['count', 'arrange', 'permutation', 'combination', 'ways', 'choose',
                          'select', 'subset', 'sequence', 'path', 'grid'],
        'probability': ['probability', 'random', 'expected', 'dice', 'coin', 'chance',
                        'independent', 'event', 'outcome'],
        'calculus': ['integral', 'derivative', 'limit', 'continuous', 'differentiate',
                     'maximum', 'minimum', 'rate'],
    }
    
    scores = {}
    for topic, keywords in topics.items():
        score = sum(1 for kw in keywords if kw in prompt_lower)
        scores[topic] = score
    
    if max(scores.values()) == 0:
        return 'other'
    return max(scores, key=scores.get)

def classify_problem_type(prompt):
    """Classify problem type."""
    prompt_lower = prompt.lower()
    
    if any(w in prompt_lower for w in ['prove', 'show that', 'demonstrate']):
        return 'proof'
    elif any(w in prompt_lower for w in ['find', 'compute', 'calculate', 'determine', 'evaluate']):
        return 'computation'
    elif any(w in prompt_lower for w in ['how many', 'count', 'number of']):
        return 'counting'
    elif any(w in prompt_lower for w in ['construct', 'give an example', 'find all']):
        return 'construction'
    else:
        return 'other'

print("Classifying topics and problem types...")
df['topic'] = df['prompt'].apply(classify_topic)
df['problem_type'] = df['prompt'].apply(classify_problem_type)

print("\nTopic distribution:")
print(df['topic'].value_counts())
print("\nProblem type distribution:")
print(df['problem_type'].value_counts())

### 2.3 Solution Quality Metrics

In [ ]:
def compute_quality_metrics(row):
    """Compute solution quality metrics."""
    completion = str(row.get('completion', ''))
    
    metrics = {}
    
    # 1. Has boxed answer (required for AIMO)
    metrics['has_boxed_answer'] = bool(re.search(r'\\boxed\{', completion))
    
    # 2. Tool usage patterns
    python_blocks = re.findall(r'to=python.*?(?=assistant|$)', completion, re.DOTALL)
    metrics['avg_code_length'] = np.mean([len(b) for b in python_blocks]) if python_blocks else 0
    
    # 3. Error handling (signs of robust code)
    metrics['has_try_except'] = 'try:' in completion and 'except' in completion
    
    # 4. Uses symbolic math (sympy)
    metrics['uses_sympy'] = 'sympy' in completion.lower()
    
    # 5. Uses numerical methods (numpy)
    metrics['uses_numpy'] = 'numpy' in completion.lower() or 'np.' in completion
    
    # 6. Reasoning steps (paragraphs of explanation)
    reasoning_blocks = re.findall(r'[A-Z][^.!?]*[.!?]', completion)
    metrics['n_reasoning_sentences'] = len(reasoning_blocks)
    
    # 7. Step-by-step structure
    metrics['has_step_structure'] = any(w in completion.lower() for w in ['step 1', 'first,', 'then,', 'finally,'])
    
    return metrics

print("Computing quality metrics...")
quality_df = df.apply(compute_quality_metrics, axis=1, result_type='expand')
df = pd.concat([df, quality_df], axis=1)
print("Done!")

print("\nQuality metrics summary:")
print(f"Has boxed answer: {df['has_boxed_answer'].mean():.1%}")
print(f"Uses sympy: {df['uses_sympy'].mean():.1%}")
print(f"Uses numpy: {df['uses_numpy'].mean():.1%}")
print(f"Has step structure: {df['has_step_structure'].mean():.1%}")

### 2.4 Novelty / Deduplication

In [ ]:
def normalize_problem(text):
    """Normalize problem text for deduplication."""
    # Remove whitespace, lowercase
    text = re.sub(r'\s+', ' ', text.lower().strip())
    # Remove LaTeX formatting variations
    text = re.sub(r'\$+', '', text)
    text = re.sub(r'\\[a-z]+\{([^}]*)\}', r'\1', text)  # \frac{a}{b} -> ab
    return text

def compute_problem_hash(text):
    """Compute hash for near-duplicate detection."""
    normalized = normalize_problem(text)
    return hashlib.md5(normalized.encode()).hexdigest()[:16]

print("Computing problem hashes for deduplication...")
df['problem_hash'] = df['prompt'].apply(compute_problem_hash)

# Find duplicates
dup_counts = df['problem_hash'].value_counts()
print(f"\nUnique problems: {len(dup_counts)}")
print(f"Duplicate groups: {(dup_counts > 1).sum()}")
print(f"Most duplicated: {dup_counts.head(10).to_dict()}")

## 3. Composite Quality Score

In [ ]:
def compute_composite_score(row):
    """Compute composite quality score (0-100)."""
    score = 0
    
    # Difficulty (prefer medium-hard, 0-25 points)
    # Sweet spot: 2-6 tool calls, 5k-30k completion length
    tool_score = min(row['n_tool_calls'] / 6, 1) * 15 if row['n_tool_calls'] <= 10 else 10
    len_score = min(row['completion_length'] / 30000, 1) * 10 if row['completion_length'] <= 50000 else 5
    score += tool_score + len_score
    
    # Quality (0-35 points)
    if row['has_boxed_answer']:
        score += 15
    if row['uses_sympy']:
        score += 10
    if row['has_step_structure']:
        score += 10
    
    # Topic diversity bonus (0-20 points)
    # Prefer underrepresented topics
    rare_topics = ['calculus', 'probability', 'geometry']
    if row['topic'] in rare_topics:
        score += 15
    else:
        score += 10
    
    # Problem type (0-20 points)
    if row['problem_type'] == 'computation':
        score += 20  # Best for training
    elif row['problem_type'] == 'counting':
        score += 18
    else:
        score += 10
    
    return min(score, 100)

print("Computing composite quality scores...")
df['quality_score'] = df.apply(compute_composite_score, axis=1)

print("\nQuality score distribution:")
print(df['quality_score'].describe())

# Show top problems
print("\nTop 10 problems by quality score:")
top_df = df.nlargest(10, 'quality_score')[['problem_id', 'topic', 'problem_type', 'n_tool_calls', 'quality_score']]
print(top_df.to_string())

## 4. Dataset Filtering & Export

In [ ]:
# Filter criteria for high-quality fine-tuning data
filtered_df = df[
    (df['has_boxed_answer'] == True) &           # Must have proper answer
    (df['n_tool_calls'] >= 1) &                   # Must use tools
    (df['n_tool_calls'] <= 15) &                  # Not too many (likely stuck)
    (df['completion_length'] >= 2000) &           # Substantial reasoning
    (df['completion_length'] <= 80000) &          # Not too long
    (df['quality_score'] >= 50)                   # Minimum quality
].copy()

print(f"Original: {len(df)} samples")
print(f"Filtered: {len(filtered_df)} samples ({len(filtered_df)/len(df):.1%})")

# Deduplicate by problem hash (keep highest quality)
filtered_df = filtered_df.sort_values('quality_score', ascending=False)
filtered_df = filtered_df.drop_duplicates(subset='problem_hash', keep='first')
print(f"After dedup: {len(filtered_df)} unique problems")

In [ ]:
# Topic balance analysis
print("\nFiltered dataset topic distribution:")
print(filtered_df['topic'].value_counts())

print("\nFiltered dataset problem type distribution:")
print(filtered_df['problem_type'].value_counts())

In [ ]:
# Stratified sampling for balanced dataset (optional)
def create_balanced_dataset(df, n_per_topic=500):
    """Create topic-balanced dataset."""
    balanced = []
    for topic in df['topic'].unique():
        topic_df = df[df['topic'] == topic].nlargest(n_per_topic, 'quality_score')
        balanced.append(topic_df)
    return pd.concat(balanced)

balanced_df = create_balanced_dataset(filtered_df, n_per_topic=1000)
print(f"\nBalanced dataset: {len(balanced_df)} samples")
print(balanced_df['topic'].value_counts())

In [ ]:
# Export curated dataset
output_path = '/kaggle/working/curated_dataset.csv'
filtered_df.to_csv(output_path, index=False)
print(f"Saved curated dataset to {output_path}")

# Also save balanced version
balanced_path = '/kaggle/working/balanced_dataset.csv'
balanced_df.to_csv(balanced_path, index=False)
print(f"Saved balanced dataset to {balanced_path}")

# Summary stats
print(f"\n" + "="*60)
print("DATASET CURATION SUMMARY")
print("="*60)
print(f"Original samples: {len(df)}")
print(f"Curated samples: {len(filtered_df)}")
print(f"Balanced samples: {len(balanced_df)}")
print(f"\nQuality score range: {filtered_df['quality_score'].min():.0f} - {filtered_df['quality_score'].max():.0f}")
print(f"Avg tool calls: {filtered_df['n_tool_calls'].mean():.1f}")
print(f"Avg completion length: {filtered_df['completion_length'].mean():.0f} chars")

In [ ]:
!ls -la /kaggle/working/*.csv